# Reading the Radar
### A hands-on guide to what weather radar sees — and misses

*Sebastián Torres · CIWRO, University of Oklahoma*  
*Clouds4Africa · July 2026*

---
**How to set up this notebook.** Wait until the top-right kernel indicator shows Python (Pyodide) and stops saying "Starting…" (the first load can take 30–60 seconds — this is normal). If the top-right says "No Kernel" instead, pick Kernel → Change Kernel → Python (Pyodide). Go to the Run menu → Run All Cells. Scroll up/down and interact with the sliders. If anything looks broken on the very first try, run that one cell again (Shift+Enter).

**How to use this notebook.** Connect what you see on a radar display to the physics and system design behind it. Each widget has three question levels — *Basic*, *A little further*, *Going deeper* — so start where you're comfortable and push as far as you can; they're meant to stretch, not to be finished. Time per widget is short, so don't expect to clear every level here; the more advanced questions are designed to be picked up again after the workshop. The notebook is yours to keep and re-run afterward.

<sub>© 2026 Sebastián Torres · Licensed under CC BY-NC 4.0 · Contact: sebas@ou.edu · Materials: https://sebastiantorr.github.io/Clouds4Africa/tree/</sub>

In [ ]:
%pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.2 MB/s eta 0:00:00


# Block 3: Range, velocity, and ambiguity
### How does the radar know how far and how fast — and where does that break?

Range and velocity both come from the *same* pulse train, and the pulse rate that helps one hurts the other. This block has three widgets. Run the setup cell once, then work through the widgets in order, since each builds on the last.

## Before we start
This block is the time-domain twin of Block 1: the same radar, sampled in time, with the ambiguities that follow. Three things to hold onto.

1. **Range and velocity come from the same pulse train.** How often the radar sends a pulse — the *PRF* — sets both how far it can see unambiguously and how fast a target it can measure.

2. **Both have a ceiling, and the ceilings fight.** Raising the PRF buys velocity but loses range; lowering it does the reverse. Their product is fixed by wavelength alone: r_max · v_max = cλ/8.

3. **Past the ceiling, the data folds.** A storm beyond the range ceiling is misplaced (range folding); one faster than the velocity ceiling reads as the wrong speed — even the wrong direction (aliasing). Neither is a malfunction; both are sampling artifacts you learn to read.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

C = 2.998e8                 # speed of light, m/s

def lam(freq_ghz):        return C/(freq_ghz*1e9)
def r_max_km(prf):        return C/(2*prf)/1000.0          # max unambiguous range (km)
def v_nyq(prf, f=2.8):    return lam(f)*prf/4.0            # Nyquist velocity (m/s)
def fold_velocity(v, vmax): return ((v + vmax) % (2*vmax)) - vmax
def product_const(f=2.8): return C*lam(f)/8000.0          # the cλ/8 ceiling, km*(m/s)

print("Block 3 ready | PRF 1000 Hz -> r_max", round(r_max_km(1000)), "km, Nyquist",
      round(v_nyq(1000),1), "m/s | product", round(r_max_km(1000)*v_nyq(1000)))

Block 3 ready | PRF 1000 Hz -> r_max 150 km, Nyquist 26.8 m/s | product 4013


## 1 — The pulse train sets the rules: range folding

A pulse must return before the next one goes out, or it's misplaced to a shorter range — a **second-trip echo**. The maximum unambiguous range is r_max = c / (2·PRF). Push a distant target past it and watch where it lands.

In [ ]:
def plot_folding(prf=1000, pulse_us=1.5, target_km=320):
    rmax = r_max_km(prf); dr = C*pulse_us*1e-6/2; prt = 1000.0/prf
    fig, (a, b) = plt.subplots(2, 1, figsize=(8, 5.4), gridspec_kw=dict(height_ratios=[1, 2]))
    for k in range(4): a.axvline(k*prt, 0.05, 0.95, color="tab:blue", lw=2)
    for k in range(3): a.axvspan(k*prt, (k+1)*prt, color="gray", alpha=0.08)
    a.annotate("", (prt, 1.15), (0, 1.15), arrowprops=dict(arrowstyle="<->"))
    a.text(prt/2, 1.25, "PRT = " + str(round(prt,2)) + " ms", ha="center", fontsize=9)
    a.set_xlim(-0.05*prt, 3*prt); a.set_ylim(0, 1.5); a.set_yticks([]); a.set_xlabel("time (ms)")
    a.set_title("pulse train @ " + str(int(prf)) + " Hz  |  r_max = " + str(round(rmax)) + " km  |  dr = " + str(round(dr)) + " m")
    r = np.linspace(0, 500, 1000)
    b.plot(r, r % rmax, color="tab:blue", lw=2, label="displayed range")
    b.plot(r, r, color="gray", ls="--", lw=1, label="true = displayed")
    b.axvline(rmax, color="tab:red", ls=":")
    dt = target_km % rmax; b.plot([target_km], [dt], "o", color="tab:red", ms=8)
    if target_km > rmax:
        b.annotate("true " + str(int(target_km)) + " km shown at " + str(round(dt)) + " km",
                   (target_km, dt), (40, dt+150), fontsize=8, color="tab:red", arrowprops=dict(arrowstyle="->"))
    b.set_xlim(0, 500); b.set_ylim(0, 560); b.set_xlabel("true range (km)"); b.set_ylabel("displayed range (km)")
    b.legend(fontsize=8, loc="upper right"); b.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(plot_folding,
         prf=IntSlider(value=1000, min=250, max=1500, step=10, description="PRF Hz"),
         pulse_us=FloatSlider(value=1.5, min=0.5, max=5, step=0.1, description="pulse µs"),
         target_km=IntSlider(value=320, min=20, max=480, step=10, description="target km"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1500, min=250, step=10), FloatSlider(val…

**Basic**

1. At 1000 Hz, what is the maximum unambiguous range? Send a target to 320 km — where does it appear?
2. Lower the PRF. Does r_max grow or shrink?

**A little further**

1. Choose one PRF that displays a storm at 300 km correctly and one that causes it to fold. For each case, record r_max​. What changed, and why did the displayed range suddenly jump?
2. A second-trip echo shows up at (true range − r_max). Reason from the pulse timing why that's exactly where it lands.

**Going deeper**

1. Suppose a radar must monitor storms out to 400 km. Using the widget, determine the highest PRF that could be used without producing second-trip echoes from that range. Why does monitoring distant storms place a limit on how frequently the radar can transmit pulses?
2. Two storms are located at 80 km and 230 km. Adjust the PRF until the distant storm folds onto the same displayed range as the nearer storm. What would the radar display show, and why does this create an interpretation problem for forecasters?

## 2 — The trade you can't escape: the Doppler dilemma

Raising the PRF buys Nyquist velocity but spends range; lowering it does the reverse. Their product r_max · v_max = cλ/8 is fixed by wavelength alone, so you only ever slide along one curve.

In [ ]:
def plot_dilemma(prf=1000, band="S"):
    fghz = 2.8 if band == "S" else 5.6
    Ks, Kc = product_const(2.8), product_const(5.6); r = np.linspace(60, 500, 400)
    plt.figure(figsize=(8, 5))
    plt.fill_between(r, Ks/r, 60, color="tab:red", alpha=0.07)
    plt.plot(r, Ks/r, color="tab:blue", lw=2.5, label="S-band")
    plt.plot(r, Kc/r, color="tab:orange", lw=1.75, ls="--", label="C-band")
    rp, vp = r_max_km(prf), v_nyq(prf, fghz)
    plt.plot([rp], [vp], "o", color="tab:blue" if band == "S" else "tab:orange", ms=9)
    plt.annotate("PRF " + str(int(prf)) + " Hz: " + str(round(rp)) + " km, " + str(round(vp)) + " m/s",
                 (rp, vp), (rp+15, vp+6), fontsize=8, arrowprops=dict(arrowstyle="->"))
    plt.plot([150], [35], "x", color="tab:red", ms=11, mew=3)
    plt.text(150, 38, "storm 35 m/s @ 150 km", color="tab:red", fontsize=8, ha="center")
    plt.xlim(0, 500); plt.ylim(0, 60); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("max unambiguous range (km)"); plt.ylabel("Nyquist velocity (m/s)")
    plt.title("the Doppler dilemma: you ride a fixed curve")
    plt.show()

interact(plot_dilemma,
         prf=IntSlider(value=1000, min=250, max=1500, step=10, description="PRF Hz"),
         band=Dropdown(options=["S", "C"], value="S", description="band"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1500, min=250, step=10), Dropdown(descri…

**Basic**

1. Slide the PRF. Does the operating point ever leave the curve?
2. Read r_max and v_max at 1000 Hz and multiply them. Does the product change as you slide?

**A little further**

1. Can you reach the marked storm (35 m/s at 150 km) at *any* single PRF? Why not, and by roughly what factor are you short?
2. Switch to C-band. Does the dilemma get easier or harder, and by about what factor? (Think cλ/8.)

**Going deeper**

1. The product r_max · v_max = cλ/8 is fixed by wavelength. Read it for S-band and C-band off the widget and confirm the ratio matches λ_S/λ_C. For C-band networks, does the shorter wavelength make the dilemma better or worse, and by how much?
2. Make it concrete with a typical 3:2 dual-PRF pair, 1200 Hz and 800 Hz. In the widget, read the Nyquist velocity at 1200 Hz (call it v₁) and at 800 Hz (v₂), and the unambiguous range at each. A dual-PRF system transmits both and combines them; the extended Nyquist is v_ext = v₁·v₂ / (v₁ − v₂). Compute v_ext. Now compare to the single-PRF curve: what single PRF would you need to reach that same velocity, and what unambiguous range would it leave you? So does dual-PRF land *on* the curve r_max·v_max = cλ/8, or beyond it — and does it finally bring the marked 35 m/s @ 150 km storm within reach?

## 3 — The artifact you'll actually see: velocity aliasing

When a storm moves faster than the Nyquist velocity, its measured velocity **aliases** into the ±v_max window (green band) — a fast outbound storm can read as inbound. On a real display that's a sharp, unphysical jump.

In [ ]:
def plot_aliasing(prf=1000, true_v=35, band="S"):
    fghz = 2.8 if band == "S" else 5.6; vmax = v_nyq(prf, fghz); vt = np.linspace(-70, 70, 1400)
    plt.figure(figsize=(8, 4.6))
    plt.axhspan(-vmax, vmax, color="tab:green", alpha=0.08)
    plt.plot(vt, fold_velocity(vt, vmax), color="tab:blue", lw=2, label="displayed")
    plt.plot(vt, vt, color="gray", ls="--", lw=1, label="true = displayed")
    d = fold_velocity(true_v, vmax); plt.plot([true_v], [d], "o", color="tab:red", ms=9)
    plt.annotate("true " + str(int(true_v)) + " -> shown " + str(round(d,1)) + ("  ALIASED" if abs(true_v) > vmax else ""),
                 (true_v, d), (true_v-5, d-24), fontsize=9, color="tab:red", arrowprops=dict(arrowstyle="->"))
    plt.xlim(-70, 70); plt.ylim(-70, 70); plt.grid(alpha=0.3); plt.legend(loc="upper left")
    plt.xlabel("true radial velocity (m/s)"); plt.ylabel("displayed velocity (m/s)")
    plt.title("velocity folding  |  Nyquist +/- " + str(round(vmax)) + " m/s @ " + str(int(prf)) + " Hz")
    plt.show()

interact(plot_aliasing,
         prf=IntSlider(value=1000, min=250, max=1500, step=10, description="PRF Hz"),
         true_v=FloatSlider(value=35, min=-70, max=70, step=1, description="true v"),
         band=Dropdown(options=["S", "C"], value="S", description="band"));

interactive(children=(IntSlider(value=1000, description='PRF Hz', max=1500, min=250, step=10), FloatSlider(val…

**Basic**

1. At 1000 Hz, what is the Nyquist velocity? Set a 35 m/s storm — what velocity is displayed?
2. Raise the PRF until 35 m/s is no longer aliased. What range did you give up (recall Widget 1)?

**A little further**

1. A display shows a sudden jump from +25 to −25 m/s across a smooth gradient. Real wind shear, or aliasing? Use the widget to make the argument.
2. A 3:2 dual-PRF scheme using 1200 and 800 Hz can extend the effective Nyquist velocity to about 64 m/s while retaining the lower PRF's range coverage. Would this allow a 35 m/s storm at 150 km to be measured without ambiguity? Explain using the widget.

**Going deeper**

1. A storm with a true radial velocity of +40 m/s is observed with a Nyquist velocity of 28 m/s. What velocity would the radar display? Show how velocity dealiasing recovers the true value.
2. Range folding changes more than an echo's location. Because the radar does not know the true distance to a second-trip echo, it computes Z using the displayed range. Would you expect the reported reflectivity to be too high, too low, or correct? Explain.

## 4 — Radar data: a hurricane's velocity field

The panels show reflectivity (left) and Doppler velocity (right) for Hurricane Isabel
(KAKQ, 18 September 2003, 0.5°), at the same scale. You don't need to read individual
gates — look at the overall patterns.

**Basic**

1. The velocity panel divides into a large region moving away from the radar (red) and one
   moving toward it (green), split by a near-zero band curving through the storm. That broad
   split is the hurricane's circulation — it's real, not an artifact. Which side is the wind
   blowing away, and which toward?
2. Knowing one side is inbound and the other outbound, where on the panel would you look for
   the strongest winds — and does the picture match a storm rotating around a center?

**A little further**

1. Scan the velocity field for places where the colors change abruptly despite the broader flow remaining smooth. Do any of those patches look more like velocity aliasing than a real wind reversal? Explain your reasoning.
2. The widget showed that aliasing happens once the wind exceeds the Nyquist velocity. So an aliased patch tells you something concrete about the wind there — what?

**Going deeper**

1. Estimate the radius of the velocity coverage. Reflectivity extends well beyond this boundary. Why can the radar detect precipitation at those greater ranges while Doppler velocity measurements stop much closer to the radar?
2. Imagine you are operating the radar during this hurricane. Your choices are:
PRF A: little aliasing but velocity coverage only to 120 km, or PRF B: velocity coverage to 240 km but substantial aliasing. Which would provide more useful information for this storm, and why? What important information would be lost with the alternative choice?

![Image](https://raw.githubusercontent.com/sebastiantorr/Clouds4Africa/main/content/images/block3_image1.png)

## Block 3 takeaway
**Range and velocity share one pulse train, and r_max · v_max = cλ/8 is a wall — a single scan gives long range or high Nyquist, not both. Folding and aliasing aren't radar faults; they're sampling artifacts you learn to read.**